In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/open-score-string-quartets


/home/dongmin/.local/share/virtualenvs/open-score-string-quartets-wd2Cnojv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import sys
import subprocess
import warnings
from typing import Union, Any, Optional
import shutil
from pathlib import Path
from collections import Counter, defaultdict
from operator import itemgetter, contains, eq # contains(A, B) == (B in A), eq(A, B) == (A == B)
from itertools import groupby
from tempfile import NamedTemporaryFile, TemporaryDirectory

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np

import xml.etree.ElementTree as ET
from ultralytics import YOLO

dformat = lambda d: json.dumps(d, indent=2)
dprint = lambda d: print(dformat(d))

In [3]:
prj_root = Path.cwd()
data_dir = prj_root / 'data'
score_dir = prj_root / 'scores'

In [4]:
data_dir, score_dir

(PosixPath('/home/dongmin/userdata/open-score-string-quartets/data'),
 PosixPath('/home/dongmin/userdata/open-score-string-quartets/scores'))

## Load metadata, xml data

In [5]:
with open(data_dir / 'scores.yaml', 'r') as f:
  data = syaml.load(f.read())

data = data.data

In [6]:
xml_paths = []

for sqid, body in data.items():
  xml_dir = score_dir / body['path']
  xml_path = xml_dir / f'sq{sqid}.musicxml'
  
  xml_paths.append(xml_path)

len(data), len(xml_paths)

(122, 122)

In [7]:
xml_paths[0]

PosixPath('/home/dongmin/userdata/open-score-string-quartets/scores/Andrée,_Elfrida/String_Quartet_in_A_major/sq7313978.musicxml')

In [15]:
TIDX = 73
test_xml = xml_paths[TIDX]

test_xml.exists(), test_xml

(True,
 PosixPath('/home/dongmin/userdata/open-score-string-quartets/scores/Maier,_Amanda/String_Quartet_in_A_major/sq7551068.musicxml'))

In [16]:
with open(test_xml, 'r') as f:
  xml = f.read()

xml_tree = ET.ElementTree(ET.fromstring(xml))

In [17]:
ref_part = xml_tree.getroot().findall('part')[0]
print_tags = ref_part.findall('.//print')

In [19]:
system_cnt = 1
systems_in_page = []

for p_t in print_tags:
  if p_t.get('new-system') == 'yes':
    system_cnt += 1
  
  elif p_t.get('new-page') == 'yes':
    systems_in_page.append(system_cnt)
    system_cnt = 1

else:
  systems_in_page.append(system_cnt)

len(systems_in_page), sum(systems_in_page)

(11, 50)

In [33]:
systems_in_page

[5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 6,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5]

In [24]:
print_tags[1].get('new-system'), print_tags[1].get('new-page')

('yes', None)